# DeepMesh com dados reais: Kaggle Data Science Bowl Cardiac

Pipeline completo no Colab:

1. baixa as ressonâncias do **Second Annual Data Science Bowl** (Kaggle): cine SAX + 2CH + 4CH, ~1.100 pacientes;
2. converte DICOM → NIfTI;
3. segmenta o miocárdio com os modelos pré-treinados de **Bai et al.** (treinados no UK Biobank): SAX e 4CH;
4. prepara os dados no formato do DeepMesh, com **controle de qualidade**;
5. treina a **reconstrução** e o **movimento** e avalia.

**Antes de rodar**
* Kaggle: crie uma conta, **verifique o telefone** e aceite as regras em
  [kaggle.com/competitions/second-annual-data-science-bowl/rules](https://www.kaggle.com/competitions/second-annual-data-science-bowl/rules).
  Em [kaggle.com/settings/api](https://www.kaggle.com/settings/api), **Generate New Token** mostra o token **uma única vez**
  numa janela: copie na hora (se fechou sem copiar, gere outro). Alternativa: *Legacy API Credentials → Create Legacy API Key*,
  que baixa um `kaggle.json` com `username` e `key`.
* GitHub: o repositório é privado. Crie um token *fine-grained* só de leitura para `eduardasaraujo/deep-mesh`
  (*Settings → Developer settings → Personal access tokens*, permissão **Contents: Read-only**).
* No Colab, ícone de **chave (Secrets)**: crie `KAGGLE_API_TOKEN` (token novo) **ou** `KAGGLE_USERNAME` + `KAGGLE_KEY`
  (chave legada), e também `GITHUB_TOKEN`. Libere o acesso deste notebook em cada um.
  Nunca escreva tokens no código.
* *Ambiente de execução → Alterar tipo → GPU (T4)*.

**Se a sessão cair:** rode as células 1 e 2 de novo e continue da etapa onde parou. Os resultados de cada etapa
e os checkpoints ficam no Google Drive (`MyDrive/deepmesh`), e todas as etapas pulam o que já foi feito.

## 1. Preparar o ambiente

In [ ]:
import os, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deepmesh'   # tudo que precisa sobreviver à sessão
REPO = '/content/deep-mesh'
os.makedirs(DRIVE, exist_ok=True)
os.environ['TQDM_DISABLE'] = '1'

def run(cmd):
    """Roda um comando no repositório mostrando a saída ao vivo."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO if os.path.isdir(REPO) else None,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'o comando falhou: {cmd}')

if not os.path.isdir(REPO):
    token = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://{token}@github.com/eduardasaraujo/deep-mesh.git', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', 'https://github.com/eduardasaraujo/deep-mesh.git'], check=True)
else:
    token = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', '-C', REPO, 'pull', '-q', f'https://{token}@github.com/eduardasaraujo/deep-mesh.git', 'main'], check=True)
run('pip install -q -r requirements.txt "kaggle>=1.8"')  # >= 1.8 aceita o token novo
run('nvidia-smi -L')

## 2. Pastas e restauração de uma sessão anterior

In [ ]:
NATIVE = '/content/native'          # NIfTI nativo + segmentações
PREPARED = '/content/dsb_prepared'  # formato do DeepMesh
NATIVE_TAR = f'{DRIVE}/dsb_native.tar'
PREPARED_TAR = f'{DRIVE}/dsb_prepared.tar'
COMMON = (f'--config configs/real.yaml --set data.root={PREPARED} output.models={DRIVE}/models '
          f'output.runs={DRIVE}/runs output.results={DRIVE}/outputs')

# os dados ficam no disco local (rápido) e uma cópia em .tar no Drive
for tar, folder in ((NATIVE_TAR, NATIVE), (PREPARED_TAR, PREPARED)):
    if os.path.exists(tar) and not os.path.isdir(folder):
        run(f'tar -xf {tar} -C /content')
        print('restaurado', folder)
run('df -h /content | tail -1')

## 3. Baixar do Kaggle

Lista os arquivos da competição e baixa para o disco local do Colab (bem mais rápido que o Drive).
Pulada se os dados já foram convertidos numa sessão anterior.

In [ ]:
def secret(name):
    try:
        return userdata.get(name)
    except Exception:  # segredo inexistente ou sem acesso liberado
        return None

if secret('KAGGLE_API_TOKEN'):       # token novo (Generate New Token)
    os.environ['KAGGLE_API_TOKEN'] = secret('KAGGLE_API_TOKEN')
elif secret('KAGGLE_KEY'):           # chave legada (kaggle.json)
    os.environ['KAGGLE_USERNAME'] = secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = secret('KAGGLE_KEY')
else:
    raise RuntimeError('crie o segredo KAGGLE_API_TOKEN (ou KAGGLE_USERNAME + KAGGLE_KEY) no Colab')
COMP = 'second-annual-data-science-bowl'

if os.path.exists(NATIVE_TAR):
    print('já convertido numa sessão anterior: pulando o download')
else:
    run(f'kaggle competitions files {COMP}')
    run(f'kaggle competitions download {COMP} -p /content/dsb_zip -q')  # -q: sem barra de progresso gigante
    run('ls -la /content/dsb_zip && df -h /content | tail -1')

## 4. DICOM → NIfTI

Lê os estudos direto do zip, um paciente por vez. Estudos sem alguma vista ou irregulares são listados em `convert.csv`.

In [ ]:
if not os.path.exists(NATIVE_TAR):
    run(f'python scripts/dsb_to_nifti.py /content/dsb_zip/*.zip --out {NATIVE} --work /content/work')
    import pandas as pd
    conv = pd.read_csv(f'{NATIVE}/convert.csv')
    print(conv.status.str.split(' ').str[0].value_counts())

## 5. Segmentação (modelos de Bai et al.)

Segmenta SAX e 4CH em todos os frames. Depois salva o NIfTI nativo + segmentações no Drive e apaga o zip.

In [ ]:
if not os.path.exists(NATIVE_TAR):
    run(f'python scripts/segment_real.py {NATIVE} --models {DRIVE}/ukbb_cardiac_models')
    run(f'tar -cf {NATIVE_TAR} -C /content native')
    run('rm -rf /content/dsb_zip /content/work')
    print('salvo em', NATIVE_TAR)

## 6. Preparar para o DeepMesh + controle de qualidade

Divide em treino/validação/teste (80/10/10) e descarta sujeitos com segmentação ruim. Veja os motivos de rejeição
e **olhe as figuras de QC**: verde = segmentação, vermelho = superfície verdadeira, ciano = template inicial.

In [ ]:
if not os.path.exists(PREPARED_TAR):
    run(f'python scripts/prepare_real.py {NATIVE} --config configs/real.yaml --set data.root={PREPARED} --workers 2')
    run(f'tar -cf {PREPARED_TAR} -C /content dsb_prepared')

import pandas as pd
qc = pd.read_csv(f'{PREPARED}/qc.csv')
print(qc.reject.fillna('ok').value_counts(), '\n')
print(qc[qc.reject.isna()].groupby('split').size())

In [ ]:
import random
from IPython.display import Image, display

subjects = sorted(os.listdir(f'{PREPARED}/train'))
for name in random.Random(0).sample(subjects, 3):
    out = f'/content/qc_{name}.png'
    run(f'python scripts/qc_figure.py {PREPARED}/train/{name} {out} --config configs/real.yaml')
    display(Image(out))

## 7. Etapa 1: reconstrução da malha ED

Checkpoints em `MyDrive/deepmesh/models`. Com `--resume`, continua de onde parou. A primeira época mostra quanto tempo
cada época leva; se faltar memória na GPU, diminua `recon.batch_size` no `--set`.

In [ ]:
run(f'python scripts/train_reconstruction.py --resume {COMMON}')

In [ ]:
run(f'python scripts/test_reconstruction.py {COMMON}')
run(f'python scripts/export_ed_meshes.py {COMMON}')

## 8. Etapa 2: movimento

In [ ]:
# as malhas ED reconstruídas ficam no disco local: refaz a exportação se a sessão foi reiniciada
first = os.path.join(PREPARED, 'train', sorted(os.listdir(f'{PREPARED}/train'))[0], 'pred_ed_vertices.npy')
if not os.path.exists(first):
    run(f'python scripts/export_ed_meshes.py {COMMON}')
run(f'python scripts/train_motion.py --resume {COMMON}')

In [ ]:
run(f'python scripts/test_motion.py {COMMON}')
run(f'python scripts/visualize.py {COMMON}')
fig_dir = f'{DRIVE}/outputs/figures'
name = sorted(os.listdir(fig_dir))[0]
for f in ('contours.png', 'meshes.png', 'curves.png'):
    display(Image(f'{fig_dir}/{name}/{f}'))